# Quick start

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/quick_start.ipynb)

You call every method in Andrey the same way: pass it your data, and read the result it returns.
This notebook simulates a small dataset, fits one method, reads the graph it learns, and draws it.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.16.0` from PyPI;
local kernels use the installed environment.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[viz]==0.16.0" pandas

In [2]:
import numpy as np
import pandas as pd

import andrey
import andrey.viz

## The data

A method reads a pandas DataFrame or a NumPy array: one row per sample, one column per variable.
Here we simulate four variables whose causes we know: `x0` causes `x1`, and `x1` and `x2` both
cause `x3`. Each effect is linear, with Gaussian noise. The columns come in an order drawn from the
seed, so their order says nothing about the causal order; `shuffle=False` keeps `x0` to `x3`.

In [3]:
EDGES = [("x0", "x1"), ("x1", "x3"), ("x2", "x3")]  # the true edges, cause first


def sample(seed: int, n: int = 2000, shuffle: bool = True) -> pd.DataFrame:
    """Draw n rows of x0..x3, the columns in an order drawn from the seed."""
    rng = np.random.default_rng(seed)
    x0 = rng.standard_normal(n)
    x1 = 0.9 * x0 + rng.standard_normal(n)
    x2 = rng.standard_normal(n)
    x3 = 0.9 * x1 + 0.9 * x2 + rng.standard_normal(n)
    frame = pd.DataFrame({"x0": x0, "x1": x1, "x2": x2, "x3": x3})
    return frame[rng.permutation(frame.columns)] if shuffle else frame


data = sample(seed=0)
data.head()

,x1,x3,x2,x0
0,0.532412,-0.105868,0.852029,0.125730
1,-0.621139,-0.925600,0.328885,-0.132105
2,-0.281320,-0.440164,-0.116862,0.640423
3,-1.505895,0.404523,0.978230,0.104900
4,-2.162436,0.814745,1.285382,-0.535669


## Fit a method

Call the method on the data. We use GES, greedy equivalence search, which looks for the graph with
the best score. Every other method is called the same way, for example `andrey.pc(data)`.

In [4]:
out = andrey.ges(data)
print(out)

ges  cpdag  |  4 nodes  |  3 edges (2 directed, 1 undirected)

  x1 -> x3
  x1 -- x0
  x2 -> x3

  score: 68.33
  score_func: bic
  penalty_discount: 1
  ebic_gamma: 1
  tie_break: rank


## Read the result

The first line names the method and the kind of graph. This one is a CPDAG: it stands for every
causal graph that the data cannot tell apart. A directed edge (`->`) points the same way in all of
them, and an undirected edge (`--`) can point either way.

GES finds the three true edges. It orients `x1 -> x3` and `x2 -> x3`, because two independent
causes of one variable leave a pattern in the data that only this orientation explains. It leaves
`x1 -- x0` undirected: `x0 -> x1` and `x1 -> x0` fit these data equally well.

You can read the same parts in code. `out.structure` is the graph: its `kind` says how to read the
edges, and `oriented_edges()` lists each edge as (source, target, type), with your column names.
`out.metadata` records the run: for GES, the score of the graph, where lower is better, and the
settings that produced it.

In [5]:
graph = out.structure
print("kind:", graph.kind)
print("edges:", graph.oriented_edges())
print("score:", round(out.metadata["score"], 2))
print("score function:", out.metadata["score_func"])

kind: cpdag
edges: [('x1', 'x3', 'directed'), ('x1', 'x0', 'undirected'), ('x2', 'x3', 'directed')]
score: 68.33
score function: bic


## Draw it

`andrey.viz.draw` renders the graph as an SVG, which the notebook shows. A directed edge ends in an
arrowhead; an undirected edge has none and is drawn in amber.

In [6]:
andrey.viz.draw(graph)

'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 360 240" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="360" height="240" fill="#fafbfc"/><line x1="140.60" y1="147.96" x2="211.24" y2="97.83" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.40,92.04 208.35,93.75 214.14,101.90" fill="#3a4658" stroke-linejoin="round"/><line x1="143.00" y1="155.48" x2="217.00" y2="155.48" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="207.00" y1="84.52" x2="143.00" y2="84.52" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="217.00,84.52 207.00,79.52 207.00,89.52" fill="#3a4658" stroke-linejoin="round"/><circle cx="130.00" cy="155.48" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="130.00" y="155.48" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x1</text><circle cx="230.00" cy="84.52" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="230.00" y="84.52" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x3</text><circle cx="130.00" cy="84.52" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="130.00" y="84.52" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x2</text><circle cx="230.00" cy="155.48" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="230.00" y="155.48" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x0</text></svg>'

## Next

- The [getting started guide](https://andrey.abel.ai/docs/guides/getting-started.html) learns a
  graph from real measurements, which `andrey.data.load_dataset("sachs")` loads, and checks it
  against a known network.
- The other [examples](https://andrey.abel.ai/docs/examples/index.html) each run one family of
  methods or one task, from data with hidden common causes to time series.
- The [API reference](https://andrey.abel.ai/docs/api.html) lists every method, the data it
  reads, and what it returns.